In [1]:
!pip -q install pandas requests matplotlib gradio

In [2]:
import requests
import sqlite3
import pandas as pd
import re
import string
import time
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import unittest
import gradio as gr

c:\UMuk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE_URL = "https://api.jikan.moe/v4"
DB_NAME = "anime_pipeline.db"

# You can change these
SEARCH_QUERY = "naruto"     # Example keyword search
MAX_PAGES = 3               # Keep moderate for demo
REQUEST_DELAY = 1.2

In [4]:
def clean_text(text):
    """
    Clean text by:
    - converting to lowercase
    - removing punctuation
    - removing HTML tags
    - removing extra spaces
    """
    if text is None or pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        return int(value)
    except:
        return default


def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except:
        return default


def classify_score_band(score):
    """
    Classifying score into bands.
    """
    
    if score < 6:
        return "low"
    elif score > 6 and score < 8:
        return "medium"
    else:
        return "high"


def classify_popularity_band(popularity):
    """
    Lower popularity number means more popular in MAL/Jikan ranking terms.
    """
    if popularity is None or popularity == 0:
        return "unknown"
    popularity = safe_int(popularity, 0)
    if popularity <= 1000:
        return "very popular"
    elif popularity <= 2000:
        return "popular"
    elif popularity <= 5000:
        return "moderate"
    else:
        return "not popular"


def classify_episode_group(episodes):
    if episodes is None or episodes == 0:
        return "unknown"
    episodes = safe_int(episodes, 0)
    if episodes <= 15:
        return "short"
    elif episodes <= 26:
        return "medium"
    else:
        return "long"


def extract_names_from_list(items):
    if not items:
        return []

    return [item["name"] for item in items if isinstance(item, dict) and "name" in item]


def count_items_in_csv(text):

    if not text:
        return 0
    return len([x.strip() for x in text.split(",") if x.strip()])


In [5]:
def fetch_anime_search(query, max_pages=1, delay=2.0):
    """
    Fetch anime data from Jikan search endpoint.
    Endpoint example: /anime?q=naruto&page=1
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/anime"
        params = {
            "q": query,
            "page": page
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime


def fetch_top_anime(max_pages=1, delay=1.2):
    """
    Fetch top anime from Jikan top endpoint.
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/top/anime"
        params = {
            "page": page
        }

        response = requests.get(url, params=params, timeout=60)
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime


In [6]:

#Logic for this function was self-devised and the code is AI-written for enhanced efficiency. 
def normalize_anime_data(raw_anime, source_label="search"):
    """
    Convert raw Jikan response into a structured DataFrame.
    """
    rows = []

    for anime in raw_anime:
        genres = extract_names_from_list(anime.get("genres", []))
        themes = extract_names_from_list(anime.get("themes", []))
        studios = extract_names_from_list(anime.get("studios", []))
        demographics = extract_names_from_list(anime.get("demographics", []))

        aired = anime.get("aired", {})
        aired_from = aired.get("from")
        aired_to = aired.get("to")

        row = {
            "mal_id": anime.get("mal_id"),
            "source_label": source_label,
            "title": anime.get("title"),
            "title_english": anime.get("title_english"),
            "type": anime.get("type"),
            "source": anime.get("source"),
            "episodes": anime.get("episodes"),
            "status": anime.get("status"),
            "airing": anime.get("airing"),
            "score": anime.get("score"),
            "scored_by": anime.get("scored_by"),
            "rank": anime.get("rank"),
            "popularity": anime.get("popularity"),
            "members": anime.get("members"),
            "favorites": anime.get("favorites"),
            "synopsis": anime.get("synopsis"),
            "rating": anime.get("rating"),
            "year": anime.get("year"),
            "season": anime.get("season"),
            "aired_from": aired_from,
            "aired_to": aired_to,
            "genres": ", ".join(genres),
            "themes": ", ".join(themes),
            "studios": ", ".join(studios),
            "demographics": ", ".join(demographics),
            "url": anime.get("url"),
            "fetched_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        }
        rows.append(row)

    return pd.DataFrame(rows)


In [7]:
#Logic for this function was self-devised and the code is AI-written for enhanced efficiency. 
def preprocess_anime_data(df):
    """
    Clean and transform anime data.
    """
    if df.empty:
        return df.copy()

    processed = df.copy()

    # Remove duplicate anime using MAL id
    processed.drop_duplicates(subset=["mal_id"], inplace=True)

    # Clean text fields
    processed["clean_title"] = processed["title"].apply(clean_text)
    processed["clean_title_english"] = processed["title_english"].apply(clean_text)
    processed["clean_synopsis"] = processed["synopsis"].apply(clean_text)

    # Text lengths
    processed["title_length"] = processed["clean_title"].apply(len)
    processed["synopsis_length"] = processed["clean_synopsis"].apply(len)

    # Binary feature
    processed["has_english_title"] = processed["title_english"].apply(
        lambda x: 0 if x is None or str(x).strip() == "" else 1
    )

    # Safe numeric conversion
    numeric_cols = ["episodes", "score", "scored_by", "rank", "popularity", "members", "favorites", "year"]
    for col in numeric_cols:
        if col == "score":
            processed[col] = processed[col].apply(lambda x: safe_float(x, 0.0))
        else:
            processed[col] = processed[col].apply(lambda x: safe_int(x, 0))

    # Engineered features
    processed["score_band"] = processed["score"].apply(classify_score_band)
    processed["popularity_band"] = processed["popularity"].apply(classify_popularity_band)
    processed["episode_group"] = processed["episodes"].apply(classify_episode_group)

    processed["genre_count"] = processed["genres"].apply(count_items_in_csv)
    processed["theme_count"] = processed["themes"].apply(count_items_in_csv)
    processed["studio_count"] = processed["studios"].apply(count_items_in_csv)
    processed["demographic_count"] = processed["demographics"].apply(count_items_in_csv)

    return processed.reset_index(drop=True)

In [8]:
def create_database(db_name):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS raw_anime (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mal_id INTEGER UNIQUE,
            source_label TEXT,
            title TEXT,
            title_english TEXT,
            type TEXT,
            source TEXT,
            episodes INTEGER,
            status TEXT,
            airing INTEGER,
            score REAL,
            scored_by INTEGER,
            rank_value INTEGER,
            popularity INTEGER,
            members INTEGER,
            favorites INTEGER,
            synopsis TEXT,
            rating TEXT,
            year INTEGER,
            season TEXT,
            aired_from TEXT,
            aired_to TEXT,
            genres TEXT,
            themes TEXT,
            studios TEXT,
            demographics TEXT,
            url TEXT,
            fetched_at TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS processed_anime (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            mal_id INTEGER,
            clean_title TEXT,
            clean_title_english TEXT,
            clean_synopsis TEXT,
            title_length INTEGER,
            synopsis_length INTEGER,
            has_english_title INTEGER,
            score_band TEXT,
            popularity_band TEXT,
            episode_group TEXT,
            genre_count INTEGER,
            theme_count INTEGER,
            studio_count INTEGER,
            demographic_count INTEGER,
            FOREIGN KEY(mal_id) REFERENCES raw_anime(mal_id)
        )
    """)

    conn.commit()
    conn.close()

In [9]:
def insert_raw_anime(df_raw, db_name):
    if df_raw.empty:
        return 0

    conn = sqlite3.connect(db_name)
    cur = conn.cursor()
    inserted = 0

    for _, row in df_raw.iterrows():
        try:
            cur.execute("""
                INSERT OR IGNORE INTO raw_anime (
                    mal_id, source_label, title, title_english, type, source, episodes, status,
                    airing, score, scored_by, rank_value, popularity, members, favorites,
                    synopsis, rating, year, season, aired_from, aired_to, genres, themes,
                    studios, demographics, url, fetched_at
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["mal_id"], row["source_label"], row["title"], row["title_english"], row["type"], row["source"],
                row["episodes"], row["status"], int(bool(row["airing"])) if pd.notna(row["airing"]) else 0,
                row["score"], row["scored_by"], row["rank"], row["popularity"], row["members"], row["favorites"],
                row["synopsis"], row["rating"], row["year"], row["season"], row["aired_from"], row["aired_to"],
                row["genres"], row["themes"], row["studios"], row["demographics"], row["url"], row["fetched_at"]
            ))
            if cur.rowcount > 0:
                inserted += 1
        except Exception as e:
            print("Raw insert error:", e)

    conn.commit()
    conn.close()
    return inserted


def insert_processed_anime(df_processed, db_name):
    if df_processed.empty:
        return 0

    conn = sqlite3.connect(db_name)
    cur = conn.cursor()
    inserted = 0

    for _, row in df_processed.iterrows():
        try:
            cur.execute("""
                INSERT INTO processed_anime (
                    mal_id, clean_title, clean_title_english, clean_synopsis,
                    title_length, synopsis_length, has_english_title, score_band,
                    popularity_band, episode_group, genre_count, theme_count,
                    studio_count, demographic_count
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["mal_id"], row["clean_title"], row["clean_title_english"], row["clean_synopsis"],
                row["title_length"], row["synopsis_length"], row["has_english_title"], row["score_band"],
                row["popularity_band"], row["episode_group"], row["genre_count"], row["theme_count"],
                row["studio_count"], row["demographic_count"]
            ))
            inserted += 1
        except Exception as e:
            print("Processed insert error:", e)

    conn.commit()
    conn.close()
    return inserted



In [10]:
def get_all_processed_anime(db_name, limit=20):
    conn = sqlite3.connect(db_name)
    query = f"""
        SELECT r.mal_id, r.title, r.title_english, r.type, r.score, r.popularity,
               r.episodes, r.year, r.season, r.genres, r.studios,
               p.score_band, p.popularity_band, p.episode_group, p.genre_count
        FROM raw_anime r
        JOIN processed_anime p
        ON r.mal_id = p.mal_id
        ORDER BY r.score DESC
        LIMIT {limit}
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


def search_anime_by_title(db_name, keyword):
    conn = sqlite3.connect(db_name)
    query = """
        SELECT r.title, r.title_english, r.type, r.score, r.popularity, r.year, r.season, r.genres, r.url
        FROM raw_anime r
        WHERE lower(r.title) LIKE ?
           OR lower(COALESCE(r.title_english, '')) LIKE ?
        ORDER BY r.score DESC
    """
    like_value = f"%{keyword.lower()}%"
    df = pd.read_sql_query(query, conn, params=(like_value, like_value))
    conn.close()
    return df


def get_average_score_by_type(db_name):
    conn = sqlite3.connect(db_name)
    query = """
        SELECT type, ROUND(AVG(score), 2) AS avg_score, COUNT(*) AS anime_count
        FROM raw_anime
        WHERE score > 0
        GROUP BY type
        ORDER BY avg_score DESC
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


def get_anime_count_by_year(db_name):
    conn = sqlite3.connect(db_name)
    query = """
        SELECT year, COUNT(*) AS anime_count
        FROM raw_anime
        WHERE year > 0
        GROUP BY year
        ORDER BY year
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


def get_top_genres(db_name, top_n=10):
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query("SELECT genres FROM raw_anime", conn)
    conn.close()

    genre_list = []
    for value in df["genres"].dropna():
        genre_list.extend([g.strip() for g in value.split(",") if g.strip()])

    top = Counter(genre_list).most_common(top_n)
    return pd.DataFrame(top, columns=["genre", "count"])


def get_top_studios(db_name, top_n=10):
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query("SELECT studios FROM raw_anime", conn)
    conn.close()

    studio_list = []
    for value in df["studios"].dropna():
        studio_list.extend([s.strip() for s in value.split(",") if s.strip()])

    top = Counter(studio_list).most_common(top_n)
    return pd.DataFrame(top, columns=["studio", "count"])


In [11]:
#The code is AI-written for enhanced efficiency. 
def run_pipeline(query=None, use_top=False, max_pages=2, db_name="anime_pipeline.db"):
    """
    Full pipeline:
    1. Fetch anime data
    2. Normalize raw data
    3. Preprocess and transform
    4. Create database
    5. Insert raw and processed data
    6. Return processed DataFrame
    """
    print("Starting pipeline...")

    if use_top:
        print("Fetching top anime...")
        raw_data = fetch_top_anime(max_pages=max_pages, delay=REQUEST_DELAY)
        source_label = "top"
    else:
        print(f"Fetching anime for query: {query}")
        raw_data = fetch_anime_search(query=query, max_pages=max_pages, delay=REQUEST_DELAY)
        source_label = "search"

    print(f"Fetched {len(raw_data)} raw records.")

    df_raw = normalize_anime_data(raw_data, source_label=source_label)
    print("Raw DataFrame shape:", df_raw.shape)

    df_processed = preprocess_anime_data(df_raw)
    print("Processed DataFrame shape:", df_processed.shape)

    create_database(db_name)

    raw_inserted = insert_raw_anime(df_raw, db_name)
    print(f"Inserted {raw_inserted} rows into raw_anime.")

    processed_inserted = insert_processed_anime(df_processed, db_name)
    print(f"Inserted {processed_inserted} rows into processed_anime.")

    print("Pipeline completed.")
    return df_raw, df_processed


In [12]:
df_raw, df_processed = run_pipeline(
    query=SEARCH_QUERY,
    use_top=False,
    max_pages=MAX_PAGES,
    db_name=DB_NAME
)

print("\nSample processed data:")
display(df_processed.head())

Starting pipeline...
Fetching anime for query: naruto
Fetched 30 raw records.
Raw DataFrame shape: (30, 27)
Processed DataFrame shape: (30, 40)
Inserted 0 rows into raw_anime.
Inserted 30 rows into processed_anime.
Pipeline completed.

Sample processed data:


C:\Users\mugha\AppData\Local\Temp\ipykernel_6828\1202507581.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


,mal_id,source_label,title,title_english,type,source,episodes,status,airing,score,...,title_length,synopsis_length,has_english_title,score_band,popularity_band,episode_group,genre_count,theme_count,studio_count,demographic_count
0,20,search,Naruto,Naruto,TV,Manga,220,Finished Airing,False,8.02,...,6,1074,1,high,very popular,long,3,1,1,1
1,16870,search,The Last: Naruto the Movie,Naruto Shippuden the Movie 7: The Last,Movie,Manga,1,Finished Airing,False,7.80,...,25,993,1,medium,very popular,short,4,0,1,1
2,28755,search,Boruto: Naruto the Movie,Boruto: Naruto the Movie,Movie,Manga,1,Finished Airing,False,7.37,...,23,981,1,medium,very popular,short,3,1,1,1
3,13667,search,Naruto: Shippuuden Movie 6 - Road to Ninja,Naruto Shippuden the Movie 6: Road to Ninja,Movie,Manga,1,Finished Airing,False,7.69,...,39,1123,1,medium,very popular,short,3,1,1,1
4,442,search,Naruto Movie 1: Dai Katsugeki!! Yuki Hime Ninp...,Naruto the Movie 1: Ninja Clash in the Land of...,Movie,Manga,1,Finished Airing,False,7.13,...,59,648,1,medium,very popular,short,3,0,1,1


In [13]:
print("Top processed anime:")
display(get_all_processed_anime(DB_NAME, limit=15))

print("Average score by type:")
display(get_average_score_by_type(DB_NAME))

print("Top genres:")
display(get_top_genres(DB_NAME, top_n=10))

print("Top studios:")
display(get_top_studios(DB_NAME, top_n=10))


Top processed anime:


,mal_id,title,title_english,type,score,popularity,episodes,year,season,genres,studios,score_band,popularity_band,episode_group,genre_count
0,53236,Road of Naruto,Road of Naruto,PV,8.39,4113,1,NaN,NaN,"Action, Fantasy",Studio Pierrot,high,moderate,short,2
1,53236,Road of Naruto,Road of Naruto,PV,8.39,4113,1,NaN,NaN,"Action, Fantasy",Studio Pierrot,high,moderate,short,2
2,1735,Naruto: Shippuuden,Naruto Shippuden,TV,8.29,15,500,2007.0,winter,"Action, Adventure, Fantasy",Studio Pierrot,high,very popular,long,3
3,1735,Naruto: Shippuuden,Naruto Shippuden,TV,8.29,15,500,2007.0,winter,"Action, Adventure, Fantasy",Studio Pierrot,high,very popular,long,3
4,20,Naruto,Naruto,TV,8.02,9,220,2002.0,fall,"Action, Adventure, Fantasy",Studio Pierrot,high,very popular,long,3
5,20,Naruto,Naruto,TV,8.02,9,220,2002.0,fall,"Action, Adventure, Fantasy",Studio Pierrot,high,very popular,long,3
6,16870,The Last: Naruto the Movie,Naruto Shippuden the Movie 7: The Last,Movie,7.80,366,1,NaN,NaN,"Action, Adventure, Fantasy, Romance",Studio Pierrot,medium,very popular,short,4
7,16870,The Last: Naruto the Movie,Naruto Shippuden the Movie 7: The Last,Movie,7.80,366,1,NaN,NaN,"Action, Adventure, Fantasy, Romance",Studio Pierrot,medium,very popular,short,4
8,13667,Naruto: Shippuuden Movie 6 - Road to Ninja,Naruto Shippuden the Movie 6: Road to Ninja,Movie,7.69,719,1,NaN,NaN,"Action, Adventure, Fantasy",Studio Pierrot,medium,popular,short,3
9,13667,Naruto: Shippuuden Movie 6 - Road to Ninja,Naruto Shippuden the Movie 6: Road to Ninja,Movie,7.69,719,1,NaN,NaN,"Action, Adventure, Fantasy",Studio Pierrot,medium,very popular,short,3


Average score by type:


,type,avg_score,anime_count
0,PV,8.39,1
1,TV,7.37,4
2,Movie,7.29,13
3,OVA,7.06,2
4,Special,7.00,7
5,CM,5.50,1


Top genres:


,genre,count
0,Action,26
1,Fantasy,23
2,Adventure,20
3,Comedy,6
4,Romance,1
5,Gourmet,1


Top studios:


,studio,count
0,Studio Pierrot,27


In [14]:
#Written by AI 
class TestAnimePipelineFunctions(unittest.TestCase):

    def test_clean_text(self):
        text = "Hello, World!!"
        self.assertEqual(clean_text(text), "hello world")

    def test_classify_score_band(self):
        self.assertEqual(classify_score_band(5.5), "low")
        self.assertEqual(classify_score_band(7.0), "medium")
        self.assertEqual(classify_score_band(8.7), "high")

    def test_classify_popularity_band(self):
        self.assertEqual(classify_popularity_band(100), "very popular")
        self.assertEqual(classify_popularity_band(1500), "popular")
        self.assertEqual(classify_popularity_band(4000), "moderate")
        self.assertEqual(classify_popularity_band(7000), "less popular")

    def test_classify_episode_group(self):
        self.assertEqual(classify_episode_group(12), "short")
        self.assertEqual(classify_episode_group(24), "medium")
        self.assertEqual(classify_episode_group(100), "long")

    def test_extract_names_from_list(self):
        items = [{"name": "Action"}, {"name": "Comedy"}]
        self.assertEqual(extract_names_from_list(items), ["Action", "Comedy"])

    def test_count_items_in_csv(self):
        self.assertEqual(count_items_in_csv("Action, Comedy, Drama"), 3)
        self.assertEqual(count_items_in_csv(""), 0)


def run_unit_tests():
    suite = unittest.TestLoader().loadTestsFromTestCase(TestAnimePipelineFunctions)
    unittest.TextTestRunner(verbosity=2).run(suite)

run_unit_tests()


test_classify_episode_group (__main__.TestAnimePipelineFunctions.test_classify_episode_group) ... ok
test_classify_popularity_band (__main__.TestAnimePipelineFunctions.test_classify_popularity_band) ... FAIL
test_classify_score_band (__main__.TestAnimePipelineFunctions.test_classify_score_band) ... ok
test_clean_text (__main__.TestAnimePipelineFunctions.test_clean_text) ... ok
test_count_items_in_csv (__main__.TestAnimePipelineFunctions.test_count_items_in_csv) ... ok
test_extract_names_from_list (__main__.TestAnimePipelineFunctions.test_extract_names_from_list) ... ok

FAIL: test_classify_popularity_band (__main__.TestAnimePipelineFunctions.test_classify_popularity_band)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\mugha\AppData\Local\Temp\ipykernel_6828\2673719973.py", line 17, in test_classify_popularity_band
    self.assertEqual(classify_popularity_band(7000), "less popular")
    ~~~~~~~~~~~~~~~~^^^^^^^^^

In [17]:
import os

def integration_test_pipeline():
    """
    Integration test:
    fetch -> normalize -> preprocess -> insert -> retrieve
    """
    test_db = "test_anime_pipeline.db"

    # Remove old test database so the test always starts fresh
    if os.path.exists(test_db):
        os.remove(test_db)

    # Fetch a very small dataset
    raw_data = fetch_anime_search(query="one piece", max_pages=1, delay=REQUEST_DELAY)
    assert len(raw_data) > 0, "No data fetched from API"

    df_raw_test = normalize_anime_data(raw_data, source_label="integration_test")
    assert not df_raw_test.empty, "Raw normalization failed"

    df_processed_test = preprocess_anime_data(df_raw_test)
    assert not df_processed_test.empty, "Preprocessing failed"

    create_database(test_db)

    raw_inserted = insert_raw_anime(df_raw_test, test_db)
    processed_inserted = insert_processed_anime(df_processed_test, test_db)

    assert raw_inserted > 0, f"Raw insert failed. Inserted: {raw_inserted}"
    assert processed_inserted > 0, f"Processed insert failed. Inserted: {processed_inserted}"

    result_df = get_all_processed_anime(test_db, limit=5)
    assert not result_df.empty, "Database retrieval failed"

    print("Integration test passed successfully.")

integration_test_pipeline()

Integration test passed successfully.


C:\Users\mugha\AppData\Local\Temp\ipykernel_6828\1202507581.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


In [18]:
def gradio_search(keyword):
    """
    Search anime titles from SQLite database.
    This demonstrates frontend-backend interaction.
    """
    if not keyword or not keyword.strip():
        return pd.DataFrame(columns=["title", "title_english", "type", "score", "popularity", "year", "season", "genres", "url"])

    results = search_anime_by_title(DB_NAME, keyword.strip())
    return results.head(20)

demo = gr.Interface(
    fn=gradio_search,
    inputs=gr.Textbox(label="Search Anime Title"),
    outputs=gr.Dataframe(label="Results"),
    title="Anime Search Interface",
    description="Search anime from the SQLite database"
)

# Uncomment to launch in Colab
# demo.launch(share=True)


# =========================
# CELL 19: EXPORT DATA
# =========================
df_processed.to_csv("processed_anime.csv", index=False)
print("Processed dataset exported as processed_anime.csv")


Processed dataset exported as processed_anime.csv


In [19]:
def get_score_band_summary(db_name):
    conn = sqlite3.connect(db_name)
    query = """
        SELECT score_band, COUNT(*) AS anime_count
        FROM processed_anime
        GROUP BY score_band
        ORDER BY anime_count DESC
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


def get_popularity_band_summary(db_name):
    conn = sqlite3.connect(db_name)
    query = """
        SELECT popularity_band, COUNT(*) AS anime_count
        FROM processed_anime
        GROUP BY popularity_band
        ORDER BY anime_count DESC
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


print("Score band summary:")
display(get_score_band_summary(DB_NAME))

print("Popularity band summary:")
display(get_popularity_band_summary(DB_NAME))

Score band summary:


,score_band,anime_count
0,medium,46
1,high,10
2,low,4


Popularity band summary:


,popularity_band,anime_count
0,moderate,30
1,popular,15
2,very popular,13
3,not popular,1
4,less popular,1
